# 01 — XBRL numeric tagger (FiNER-139)

Fine-tunes `nlpaueb/sec-bert-base` for token classification over 139 XBRL
concepts, so KPI extraction can decide whether a given number *is* `Revenues` or
`NetIncomeLoss` rather than matching a nearby label with a regex.

| | |
|---|---|
| Base | `nlpaueb/sec-bert-base` (plain — not `-num`/`-shape`, so the paper comparison is clean) |
| Data | `nlpaueb/finer-139` — 900,384 / 112,494 / 108,378 |
| Task | token classification, 279 labels (139 concepts × B-/I-, plus `O`) |
| T4 | batch 32, max_len 256, fp16, 2 epochs, lr 3e-5 |

**Published reference (not this repo's result):** the FiNER-139 paper reports
89.2% micro-F1 for `sec-bert-base` on the full splits. If you train on a subset,
your number is not comparable to it — quote your own baseline delta instead.

Two ways this trains happily while producing meaningless numbers:
- **Label alignment** — only the first sub-word of a word carries the tag;
  continuations and specials get `-100`.
- **Scoring** — `seqeval` at span level. Token accuracy is meaningless because
  `O` dominates; predicting it everywhere scores above 95%.

In [ ]:
# Confirm we actually have the T4 this recipe is written for.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
print(f"torch {torch.__version__} | {torch.cuda.get_device_name(0)} | "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Checkpoints MUST live somewhere that survives the VM (section 5.5).
# /content is ephemeral - it vanishes with the runtime, which is exactly the
# failure checkpointing exists to defend against.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/affa'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('checkpoints ->', DRIVE_ROOT)

In [ ]:
# Clone or update the repo, and verify it is current. Re-running this notebook
# from the top after a disconnect must not silently train an old revision.
import os, subprocess

REPO_URL = 'https://github.com/abhinaba01/agentic-financial-filing-analysis.git'
REPO_DIR = '/content/agentic-financial-filing-analysis'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all'], check=True)

local  = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True).stdout.strip()
remote = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '@{u}'],
                        capture_output=True, text=True).stdout.strip()

if remote and local != remote:
    print(f'repo is BEHIND origin (local {local[:8]} != remote {remote[:8]})')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
    print('pulled; RESTART THE RUNTIME so the new code is imported')
else:
    print(f'repo is current at {local[:8]}')

os.chdir(REPO_DIR)

In [ ]:
# datasets<4.0 is REQUIRED, not a preference: finer-139, financial_phrasebank
# and finqa are loading-script datasets, and datasets>=4.0 removed script
# execution entirely. The parquet mirrors are NOT equivalent - at least one is
# deduplicated, which changes the splits and breaks comparability.
%pip install -q -e ".[train,eval]"
%pip install -q "datasets>=2.19,<4.0"

import datasets, transformers
print('datasets', datasets.__version__, '| transformers', transformers.__version__)
assert int(datasets.__version__.split('.')[0]) < 4, (
    'datasets>=4.0 cannot execute loading scripts; pin datasets>=2.19,<4.0'
)

In [ ]:
# Run configuration. These values are written into the checkpoint directory and
# re-checked on resume - changing one after a crash invalidates the run.
SEED          = 42
TRAIN_SAMPLES = 200_000   # None = all 900k (does not fit one 12h T4 session)
EVAL_SAMPLES  = 10_000
MAX_LENGTH    = 256
BATCH_SIZE    = 32
EPOCHS        = 2
LR            = 3e-5
SAVE_STEPS    = 500       # ~15-20 min of training on a T4 at this batch size

CKPT_DIR = f'{DRIVE_ROOT}/xbrl_tagger'
print(CKPT_DIR)

## Checkpointing and resume

Colab runtimes disconnect, get recycled, and hit idle timeouts. Everything below
is built so a crash costs minutes, not the whole run.

**What resume restores:** model weights, optimizer moments, LR-scheduler
position, RNG state, global step, and dataloader position. That is why we resume
rather than "just train again from the saved weights" — restarting the optimizer
and the LR schedule from scratch is a *different run*, and its loss curve will
not join up with the first half.

**The cell below is idempotent.** Re-run it after a crash and it resumes
automatically, with no code edit.

**Determinism is a precondition.** `SEED`, `TRAIN_SAMPLES` and `EVAL_SAMPLES` are
written into the checkpoint directory as JSON, and the resume path *refuses* to
continue if they no longer match. Changing any of them after a crash means the
global step now points into different data and the resumed run is silently
meaningless (anti-pattern #14).

**Disk:** a full checkpoint is roughly 3–4× model size — fp32 weights plus two
AdamW moments — so `save_total_limit=2` is required, not tidiness, against
Drive's 15GB free tier. `save_steps` is set for ~15–20 minutes of training, not
per epoch: an epoch here is 40+ minutes and a disconnect at minute 39 loses all
of it.

In [ ]:
# Idempotent: re-run after a crash and it resumes from the last checkpoint.
!python training/train_xbrl_tagger.py \
    --output-dir "{CKPT_DIR}" \
    --seed {SEED} \
    --train-samples {TRAIN_SAMPLES} \
    --eval-samples {EVAL_SAMPLES} \
    --max-length {MAX_LENGTH} \
    --batch-size {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --learning-rate {LR} \
    --save-steps {SAVE_STEPS}

## Test the resume path — do not assume it

Untested resume logic is usually broken resume logic, and the moment you find
out is the moment you have already lost the run.

1. Run the training cell above and let it write at least two checkpoints.
2. **Runtime → Interrupt execution** (or just let the runtime die).
3. Re-run the training cell *unchanged*.

What you should see: `resuming from .../checkpoint-N`, and the loss continuing
from where it stopped rather than restarting near its initial value. If step
numbering restarts at 0, resume is not working — fix that before starting the
real run.

## Evaluate against a measured baseline

The harness reports the per-concept breakdown as well as micro-F1. That is not
optional: the tag distribution is severely skewed, and a headline number close to
the paper's can sit on top of a model that learned six concepts and ignored the
other 133.

In [ ]:
# Scores the TEST split once, against the base encoder as a measured floor.
# The paper's 89.2% appears in the output under its own heading, labelled as a
# published figure produced under different conditions.
!affa-eval xbrl \
    --model "{CKPT_DIR}/final" \
    --limit 5000 \
    --output eval_results/xbrl.json

import json
print(json.dumps(json.load(open('eval_results/xbrl.json'))['metrics'], indent=2))

In [ ]:
# Push the model and a card carrying the REAL numbers and the subset size.
# A model card with aspirational numbers is worse than no card.
from huggingface_hub import notebook_login
notebook_login()

HUB_ID = 'YOUR_USERNAME/affa-xbrl-tagger'

card = """---
license: apache-2.0
tags: [finance, sec-filings, affa]
---

# affa-xbrl-tagger

Fine-tuned for the Agentic Financial Filing Analyst.

Token classification over FiNER-139 (139 US-GAAP concepts, B-/I- tagging).

## Measured results

Fill these in from the evaluation cell above. Report the **test** split score,
the **baseline measured on the same data with the same protocol**, and the
training subset size. Do not paste a number from a paper here.

| metric | this model | baseline | notes |
|---|---:|---:|---|
| (fill in) | | | |

- Training subset: `TRAIN_SAMPLES` (state the number actually used)
- Seed: `SEED`
- Checkpoint selected on: validation split
- Test split touched: once

## Not financial advice

Research and educational use only.
"""

import pathlib
pathlib.Path(f'{CKPT_DIR}/final/README.md').write_text(card, encoding='utf-8')
print('model card written; review it before pushing')